# 22 — Package & Distribute

Takes the best model produced by the training pipeline and packages it for distribution.

## What this notebook does

```
Best distilled student (NB21)  —OR—  iterative (NB20)  —OR—  fine-tune (NB17)
         ↓
  1. Verify & select best checkpoint
         ↓
  2. Quantize to 4-bit MLX  →  release/aro-coder-4bit/
         ↓
  3. Smoke-test the quantized model
         ↓
  4a. MLX server  →  startup script  (local OpenAI-compatible API)
  4b. Ollama      →  Modelfile + aro-coder:latest  (local chat)
  4c. HuggingFace →  push to ARO-Lang/aro-coder-4bit  (public)
         ↓
  5. `aro ask` CLI integration  →  version manifest for download checks
```

## Setup

In [ ]:
import sys, importlib
from pathlib import Path
_cfg_dir = Path('.').resolve()
if _cfg_dir not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *
print(f'Config loaded | model={MODEL_ID}')

import json, re, subprocess, shutil, time

RELEASE_DIR.mkdir(parents=True, exist_ok=True)

QUANT_DIR   = RELEASE_DIR / 'aro-coder-4bit'

# HuggingFace repo (change to your org/username)
HF_REPO_ID  = 'ARO-Lang/aro-coder-4bit'

# Ollama model name
OLLAMA_NAME = 'aro-coder'

print(f'Models dir:  {MODELS_DIR}')
print(f'Release dir: {RELEASE_DIR}')
print(f'HF repo:     {HF_REPO_ID}')
print(f'Ollama name: {OLLAMA_NAME}')

## Step 1 — Select best checkpoint

Priority: distilled student (NB21) > DPO (NB18) > iterative (NB20) > fine-tune (NB17) > warm-start (NB05).

In [ ]:
def find_best_fused_model():
    """Return path to the best available fused model directory.

    Priority (smaller/faster preferred when known-healthy):
      1. Distilled student (NB12)     — models/distill/student/fused       (8B, ~3x faster than 30B)
      2. DPO model (NB12)             — models/dpo/fused                   (30B teacher, full SFT+DPO)
      3. Iterative loop models (NB12) — models/iterative/round_*/fused     (30B, multi-round refined)
      4. Fine-tune models (NB12)      — models/finetune/round_*/fused      (30B, plain SFT)
      5. Warm-start adapter (NB05)    — fused on the fly
      6. Raw base model               — no fine-tuning at all

    The distilled 8B student is preferred *because* it is smaller and faster.
    Safety contract: NB12 cell distill_07 runs a post-fusion validator and
    DELETES the fused student directory if it has collapsed (e.g. emits only
    `!`). So if `models/distill/student/fused/` exists at all, NB12 has
    confirmed it produces sensible output. If validation has not run on the
    current student (e.g. it was created before the validator existed), delete
    that directory manually so this selector falls through to DPO.
    """

    def _highest_fused(parent, label):
        if not parent.exists():
            return None
        rounds = sorted(
            [d for d in parent.glob('round_*/fused') if (d / 'config.json').exists()],
            key=lambda p: int(re.search(r'round_(\d+)', str(p)).group(1))
        )
        if rounds:
            best = rounds[-1]
            round_n = re.search(r'round_(\d+)', str(best)).group(1)
            print(f'Found {label} model: round {round_n} → {best}')
            return best, f'{label}_round_{round_n}'
        return None

    # 1. Distilled student (NB12) — smaller/faster, validated by NB12 itself
    distill_fused = DISTILL_MODELS_DIR / 'student' / 'fused'
    if distill_fused.exists() and (distill_fused / 'config.json').exists():
        print(f'Found distilled student (validated by NB12): {distill_fused}')
        return distill_fused, 'distill_student'

    # 2. DPO model (NB12) — fallback when no healthy student exists
    dpo_fused = MODELS_DIR / 'dpo' / 'fused'
    if dpo_fused.exists() and (dpo_fused / 'config.json').exists():
        print(f'Found DPO model: {dpo_fused}')
        return dpo_fused, 'dpo'

    # 3. Iterative loop models (NB12)
    result = _highest_fused(ITERATIVE_MODELS_DIR, 'iterative')
    if result:
        return result

    # 4. Fine-tune models (NB12)
    result = _highest_fused(FINETUNE_MODELS_DIR, 'finetune')
    if result:
        return result

    # 5. Fuse warm-start adapter on the fly
    warm = resolve_warm_adapter()
    if warm:
        fused_path = RELEASE_DIR / 'warm_start_fused'
        if not (fused_path / 'config.json').exists():
            print(f'No fused rounds found — fusing warm-start adapter...')
            cmd = [
                sys.executable, '-m', 'mlx_lm', 'fuse',
                '--model',        local_model_path(MODEL_ID),
                '--adapter-path', warm,
                '--save-path',    str(fused_path),
            ]
            subprocess.run(cmd, check=True)
        else:
            print(f'Using cached warm-start fused model: {fused_path}')
        return fused_path, 'warm_start_fused'

    # 6. Last resort
    print(f'WARNING: No fine-tuned model found — will quantize the base model.')
    return MODEL_ID, 'base'


SOURCE_MODEL, SOURCE_LABEL = find_best_fused_model()
print(f'\nSource model : {SOURCE_MODEL}')
print(f'Source label : {SOURCE_LABEL}')
print(f'Base model:    {MODEL_ID}')

## Step 2 — Quantize to 4-bit MLX

In [ ]:
def _is_already_quantized(source):
    """Robust check: model is already quantized if config.json declares it OR
    safetensors files contain `.scales`/`.biases` tensors (the MLX naming
    convention for quantized weights). The fused output produced by
    `mlx_lm fuse` on a 4-bit base writes quantized weights without always
    populating `quantization_config` in config.json — we have to peek at the
    actual tensor names too.
    """
    src = Path(source)

    cfg_path = src / 'config.json'
    if cfg_path.exists():
        try:
            with open(cfg_path) as f:
                cfg = json.load(f)
            if 'quantization_config' in cfg or 'quantization' in cfg:
                return True
        except Exception:
            pass

    # Inspect the first safetensors file for quantized-tensor naming.
    sf_files = sorted(src.glob('*.safetensors'))
    if sf_files:
        try:
            import struct as _struct
            with open(sf_files[0], 'rb') as f:
                header_len = _struct.unpack('<Q', f.read(8))[0]
                header = json.loads(f.read(header_len))
            keys = list(header.keys())
            if any(k.endswith('.scales') or k.endswith('.biases') for k in keys):
                return True
        except Exception:
            pass

    return False


def _peek_tensor_names(source, n=20):
    """Return up to n tensor names from the first safetensors in source."""
    src = Path(source)
    sf_files = sorted(src.glob('*.safetensors'))
    if not sf_files:
        return []
    try:
        import struct as _struct
        with open(sf_files[0], 'rb') as f:
            header_len = _struct.unpack('<Q', f.read(8))[0]
            header = json.loads(f.read(header_len))
        keys = [k for k in header.keys() if k != '__metadata__']
        return keys[:n]
    except Exception as e:
        return [f'(could not read tensor names: {e})']


def _source_matches_quant(source, quant_dir):
    """Check if the quantized model was built from the current source AND
    whether the source has been modified since.

    The previous implementation only compared the source path string.
    That missed the case where NB21 re-fused the same student/fused dir
    with newer weights — the path stayed identical but the contents
    changed. NB22 then "matched" the stale quant and shipped round-N-1
    weights even after round-N training. Adding an mtime check closes
    this hole.
    """
    marker = quant_dir / '.source_model'
    if not marker.exists():
        return False
    if marker.read_text().strip() != str(source):
        return False
    # Compare source's newest safetensors mtime against the marker's mtime.
    src = Path(source)
    src_safetensors = sorted(src.glob('*.safetensors'))
    if not src_safetensors:
        return True  # nothing to compare; fall through to copy path
    newest_src = max(f.stat().st_mtime for f in src_safetensors)
    if newest_src > marker.stat().st_mtime:
        print(f'  source {source} has newer weights than the marker '
              f'(src={newest_src:.0f} > marker={marker.stat().st_mtime:.0f}) — re-quantizing')
        return False
    return True


def _validate_safetensors_headers(directory):
    """Validate safetensors header lengths.

    The previous implementation rewrote the 8-byte length prefix whenever JSON
    parsing of the declared header failed. JSON parsing can fail for benign
    reasons (e.g., an unfamiliar key encoding), and rewriting the prefix in
    those cases corrupts an otherwise-valid file. We now only *report* mismatches
    and never modify the file. If a real corruption is detected, re-quantize
    rather than patching bytes in place.
    """
    import struct as _struct
    directory = Path(directory)
    for sf in sorted(directory.glob('*.safetensors')):
        size = sf.stat().st_size
        with open(sf, 'rb') as f:
            len_data = f.read(8)
        if len(len_data) < 8:
            print(f'  WARN {sf.name}: file too small to contain a header')
            continue
        declared = _struct.unpack('<Q', len_data)[0]
        if declared <= 0 or declared > size - 8:
            print(f'  WARN {sf.name}: header length {declared} inconsistent with file size {size} '
                  f'— consider re-quantizing this directory')
            continue
        with open(sf, 'rb') as f:
            f.seek(8)
            header_bytes = f.read(declared)
        try:
            json.loads(header_bytes)
        except Exception as e:
            print(f'  WARN {sf.name}: header at declared length {declared} did not parse as JSON ({e}) '
                  f'— leaving file untouched (re-quantize if loading fails)')


def quantize(source, dest, q_bits=4, q_group_size=64):
    """Quantize source model to q_bits and save to dest.

    If the source is already quantized (e.g. fused from a 4-bit base),
    copies it directly instead of re-quantizing. The detection looks at both
    config.json keys AND safetensors tensor names so we do not try to
    re-quantize already-4-bit weights (which `mlx_lm convert` rejects).

    Re-quantizes if the source model has changed since the last run.
    """
    dest = Path(dest)

    # Check if existing quantized model matches current source
    if (dest / 'config.json').exists():
        if _source_matches_quant(source, dest):
            print(f'Quantized model already exists at {dest} (matches source) -- skipping.')
            _validate_safetensors_headers(dest)
            return dest
        else:
            print(f'Source model changed -- removing stale quantized model at {dest}')
            shutil.rmtree(dest)

    # If the fused model was built from a quantized base, it is already
    # quantized. Re-quantizing would fail or produce garbage -- just copy it.
    already_q = _is_already_quantized(source)
    if already_q:
        print(f'Source is already quantized -- copying {source} -> {dest}')
        dest.mkdir(parents=True, exist_ok=True)
        src = Path(source)
        for f in src.iterdir():
            if f.is_file():
                shutil.copy2(f, dest / f.name)
        size_gb = sum(f.stat().st_size for f in dest.rglob('*') if f.is_file()) / 1e9
        print(f'Copied  |  size: {size_gb:.1f} GB  |  saved to {dest}')
    else:
        # mlx_lm convert refuses to write to a path that already exists, so
        # remove any stale destination directory left by a prior partial run
        # and DO NOT pre-create it. The convert command creates the dir
        # itself.
        if dest.exists():
            print(f'Removing stale {dest} so mlx_lm convert can create it fresh')
            shutil.rmtree(dest, ignore_errors=True)
        print(f'Quantizing {source} -> {dest} ({q_bits}-bit, group={q_group_size})...')
        print(f'  Tensor names sample: {_peek_tensor_names(source, n=8)}')

        cmd = [
            sys.executable, '-m', 'mlx_lm', 'convert',
            '--hf-path',      str(source),
            '--mlx-path',     str(dest),
            '--quantize',
            '--q-bits',       str(q_bits),
            '--q-group-size', str(q_group_size),
        ]
        t0 = time.time()
        # Stream stdout/stderr live so the actual mlx_lm error is visible
        # immediately instead of being trapped by capture_output. We tee to
        # a sibling log file (not inside dest, since dest must not yet exist).
        log_path = dest.parent / f'{dest.name}_convert.log'
        log_path.parent.mkdir(parents=True, exist_ok=True)
        captured_lines = []
        with open(log_path, 'w') as logf:
            proc = subprocess.Popen(
                cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, bufsize=1,
            )
            assert proc.stdout is not None
            for line in proc.stdout:
                print(f'  [mlx_lm] {line}', end='')
                logf.write(line)
                captured_lines.append(line)
            proc.wait()
        elapsed = time.time() - t0

        if proc.returncode != 0:
            tail = ''.join(captured_lines[-60:])
            shutil.rmtree(dest, ignore_errors=True)
            print('=' * 70)
            print('mlx_lm convert FAILED')
            print('=' * 70)
            print(f'Exit code: {proc.returncode}')
            print(f'Command:   {" ".join(cmd)}')
            print(f'Log:       {log_path}')
            print(f'Source:    {source}')
            try:
                _src = Path(source)
                _files = sorted(p.name for p in _src.iterdir() if p.is_file())
                print(f'Source dir contains {len(_files)} files: {_files[:8]}'
                      + (' ...' if len(_files) > 8 else ''))
                _cfg = _src / 'config.json'
                if _cfg.exists():
                    with open(_cfg) as _cf:
                        _cdat = json.load(_cf)
                    _arch = _cdat.get('architectures') or _cdat.get('model_type')
                    print(f'  config.json architecture: {_arch}')
                    print(f'  has quantization_config:  {"quantization_config" in _cdat}')
                    print(f'  has quantization:         {"quantization" in _cdat}')
                _names = _peek_tensor_names(source, n=20)
                print(f'  tensor name sample (first 20):')
                for _n in _names:
                    print(f'    {_n}')
            except Exception as _e:
                print(f'  (could not introspect source: {_e})')
            print('-' * 70)
            print('Last 60 lines of mlx_lm output:')
            print(tail or '(empty)')
            print('=' * 70)
            print('Common causes:')
            print('  1. Source already quantized (look for .scales/.biases above) — delete')
            print('     models/.../fused/ and re-fuse, OR force the copy path manually.')
            print('  2. mlx_lm version mismatch — `pip install -U mlx-lm` and retry.')
            print('  3. Out of memory — quantizing 30B needs ~32 GB free RAM.')
            print('  4. Source path missing or partial — check the file listing above.')
            print('  5. MoE architecture (qwen3_moe) — older mlx_lm cannot quantize MoE')
            print('     models; upgrade mlx-lm or quantize a dense student instead.')
            raise RuntimeError('Quantization failed — see diagnostics above.')

        size_gb = sum(f.stat().st_size for f in dest.rglob('*') if f.is_file()) / 1e9
        print(f'Done in {elapsed:.0f}s  |  size: {size_gb:.1f} GB  |  saved to {dest}')

    # Write source marker so we know what model this was built from
    (dest / '.source_model').write_text(str(source))

    # Validate (read-only) safetensors headers
    _validate_safetensors_headers(dest)

    return dest


QUANT_DIR = quantize(SOURCE_MODEL, QUANT_DIR)
print(f'\nQuantized model ready: {QUANT_DIR}')

record_stage('quantize', 'passed',
             fingerprint=_quant_fingerprint(QUANT_DIR),
             source_model=str(SOURCE_MODEL), source_label=SOURCE_LABEL)

## Step 3 — Smoke test

Load the quantized model and run a quick ARO generation to verify it works.

Save point: the result is checkpointed in `release_state.json` keyed on a
fingerprint of the quantized artifacts. Re-running the notebook skips a
smoke test that already passed for the same artifacts; set
`ARO_FORCE_GATES=1` for an explicit, audit-logged re-run.

In [ ]:
import csv, struct
from datetime import date as _date

kb = load_knowledge()
SYSTEM_PROMPT = build_system_prompt(kb)

load_fn, generate_fn, make_sampler_fn = ensure_mlx_lm()

# ── Save point: skip if the smoke test already passed for these artifacts ──
_smoke_fp = _quant_fingerprint(QUANT_DIR)
if stage_completed('smoke_test', _smoke_fp):
    _prev = stage_result('smoke_test')
    results   = _prev.get('results', [])
    passed    = _prev.get('passed', 0)
    pass_rate = _prev.get('pass_rate', 0.0)
    print(f'Smoke test already PASSED for these artifacts '
          f'({pass_rate:.0%} at {_prev.get("at", "?")}) — skipping.')
    print(f'  (set ARO_FORCE_GATES=1 to force a re-run; state: {RELEASE_STATE_PATH})')
else:
    print(f'Loading quantized model from {QUANT_DIR}...')
    model, tokenizer = load_fn(str(QUANT_DIR))
    print('Model loaded.\n')

    # ── Configurable smoke test threshold ────────────────────────────────────
    MIN_SMOKE_PASS_RATE = 0.30  # At least 30% of prompts must produce valid ARO code

    TEST_PROMPTS = [
        'Write an ARO feature set that retrieves a user by ID from a repository and returns it as an OK response.',
        'How do I emit a custom event in ARO and handle it in another feature set?',
        'Write a minimal ARO Application-Start that starts an HTTP server.',
        'Write a feature set named listOrders that retrieves all orders from the order-repository.',
        'Write an ARO feature set that creates a new product from the request body.',
    ]

    # ── Run aro check on generated code blocks ───────────────────────────────
    import re, subprocess, tempfile

    def _extract_aro_blocks(text):
        pattern = r'```aro\n([\s\S]*?)```'
        return [m.group(1).strip() for m in re.finditer(pattern, text) if m.group(1).strip()]

    def _aro_check(code):
        """Run aro check on code, return (passed, error_msg)."""
        with tempfile.TemporaryDirectory() as tmp:
            p = Path(tmp) / 'main.aro'
            p.write_text(code)
            aro_bin = shutil.which('aro') or 'aro'
            try:
                r = subprocess.run([aro_bin, 'check', tmp], capture_output=True, text=True, timeout=10)
                if r.returncode == 0:
                    return True, ''
                return False, (r.stderr or r.stdout).strip()[:200]
            except Exception as e:
                return False, str(e)[:200]

    results = []
    passed = 0

    for prompt in TEST_PROMPTS:
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': prompt},
        ]
        text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        tokens = tokenizer.encode(text)
        output = generate_fn(
            model, tokenizer,
            prompt=tokens,
            max_tokens=400,
            sampler=make_sampler_fn(temp=0.2),
            verbose=False,
        )

        # Check for valid ARO code
        blocks = _extract_aro_blocks(output)
        has_code = len(blocks) > 0
        check_pass = False
        check_error = ''
        if blocks:
            check_pass, check_error = _aro_check('\n\n'.join(blocks))

        if check_pass:
            passed += 1

        results.append({
            'prompt': prompt,
            'response_preview': output.strip()[:200],
            'has_code_block': has_code,
            'aro_check_pass': check_pass,
            'check_error': check_error,
        })

        status = 'PASS' if check_pass else ('NO CODE' if not has_code else 'FAIL')
        print(f'[{status}] {prompt[:60]}...')
        if not check_pass:
            preview = output.strip()[:120].replace('\n', ' ')
            print(f'       Response: {preview}')
        print()

    pass_rate = passed / len(TEST_PROMPTS)
    print(f'\nSmoke test: {passed}/{len(TEST_PROMPTS)} passed ({pass_rate:.0%})')

    # ── CSV export ───────────────────────────────────────────────────────────
    _run_dir = Path('.') / 'run' / _date.today().isoformat()
    _run_dir.mkdir(parents=True, exist_ok=True)
    csv_path = _run_dir / '22_package.csv'
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['prompt', 'response_preview', 'has_code_block', 'aro_check_pass', 'check_error'])
        writer.writeheader()
        writer.writerows(results)
    print(f'CSV saved: {csv_path}')

    # ── Gate: abort if below threshold ───────────────────────────────────────
    del model

    if pass_rate < MIN_SMOKE_PASS_RATE:
        record_stage('smoke_test', 'failed', fingerprint=_smoke_fp,
                     pass_rate=pass_rate, passed=passed,
                     total=len(TEST_PROMPTS), results=results)
        raise RuntimeError(
            f'Smoke test FAILED: {passed}/{len(TEST_PROMPTS)} passed ({pass_rate:.0%}), '
            f'minimum required is {MIN_SMOKE_PASS_RATE:.0%}. '
            f'The model cannot generate valid ARO code. '
            f'Check the distillation (NB12) and fine-tune (NB12) stages. '
            f'Gate state recorded in {RELEASE_STATE_PATH}.'
        )

    record_stage('smoke_test', 'passed', fingerprint=_smoke_fp,
                 pass_rate=pass_rate, passed=passed,
                 total=len(TEST_PROMPTS), results=results)
    print(f'\nSmoke test PASSED ({pass_rate:.0%} >= {MIN_SMOKE_PASS_RATE:.0%} threshold)')
    print('Proceeding to distribution...')


## Step 4 — Promotion gate (100-prompt sweep, fused vs quantized)

Runs the full 100-prompt evaluation sweep against **both** the
pre-quantization fused model and the quantized model, compares the metrics
side-by-side, and raises if the quantized model doesn't hit the thresholds:

- **Reply rate** ≥ 50% (model must produce content, not empty `<think></think>`)
- **Empty-think collapse** ≤ 20% of prompts
- **Syntax pass rate** ≥ 40% of code blocks
- **Tool-name leakage** ≤ 2%, **URL contamination** ≤ 5%
- **Quantization degradation**: quantized may not lose more than 10 points of
  syntax pass rate or reply rate vs the fused model

The last gate catches quantization regressions directly instead of hoping the
absolute thresholds happen to hit them. Both sets of metrics land in
`promotion_gate.json`. When the source model is itself already quantized
(fused-from-4-bit copy path), the comparison is a no-op and only the
quantized sweep runs.

Save point: the gate result (metrics + breaches) is checkpointed in
`release_state.json`. A gate that already passed for the same quantized
artifacts is skipped on re-run; a failed gate leaves a structured failure
record. `ARO_FORCE_GATES=1` forces an explicit, audit-logged re-run instead
of a silent one that could flip an intermittent failure to a pass by chance.


In [ ]:
# ── Promotion gate: 100-prompt sweep, fused vs quantized ───────────────────
# Runs the evaluation prompts through BOTH the pre-quantization fused model
# and the quantized model, compares metrics side-by-side, and gates the
# HF/Ollama/MLX distribution steps below on:
#   - reply rate (must produce content)
#   - empty-think collapse rate (no <think></think> + EOS)
#   - syntax pass rate (code blocks that pass `aro check`)
#   - tool-name leakage (model emitting read_file(...) etc as ARO)
#   - URL/HTML contamination (model leaking corpus pollution)
#   - quantization degradation (quantized must not regress beyond the drop
#     thresholds vs fused — previously such regressions were only caught if
#     the absolute gates happened to hit them)

import json as _json, re as _re, subprocess as _sp, tempfile as _tmp
from pathlib import Path as _Path

EVAL_PROMPTS_PATH = _Path(__file__).resolve().parent.parent / 'eval_prompts.json' \
    if '__file__' in dir() else _Path('/Users/kris/Projects/ARO/ARO-Lang/Train/eval_prompts.json')
if not EVAL_PROMPTS_PATH.exists():
    for _cand in [
        _Path.cwd().parent / 'eval_prompts.json',
        _Path.cwd() / 'eval_prompts.json',
        _Path('/Users/kris/Projects/ARO/ARO-Lang/Train/eval_prompts.json'),
    ]:
        if _cand.exists():
            EVAL_PROMPTS_PATH = _cand
            break

GATE_REPLY_RATE_MIN     = 0.50   # at least 50% must produce non-empty content
GATE_EMPTY_THINK_MAX    = 0.20   # at most 20% may collapse to empty <think></think>
GATE_SYNTAX_PASS_MIN    = 0.40   # at least 40% of code blocks must pass aro check
GATE_TOOL_LEAK_MAX      = 0.02   # at most 2% of replies may contain tool-call leakage
GATE_URL_CONTAM_MAX     = 0.05   # at most 5% of replies may contain markdown image URLs

# Quantized-vs-fused degradation thresholds (absolute percentage points).
GATE_QUANT_SYNTAX_DROP_MAX = 0.10   # quantized syntax pass may drop ≤ 10 pts vs fused
GATE_QUANT_REPLY_DROP_MAX  = 0.10   # quantized reply rate may drop ≤ 10 pts vs fused

_eval_prompts = _json.loads(EVAL_PROMPTS_PATH.read_text())
print(f'Promotion gate: {len(_eval_prompts)} prompts per model')

_THINK_RE = _re.compile(r'<think>[\s\S]*?</think>', _re.S)
_FENCE_RE = _re.compile(r'```aro\s*\n(.*?)```', _re.S)
_TOOL_RE  = _re.compile(r'\b(read_file|write_file|edit_file|aro_check|aro_run|aro_test|create_plugin|list_files)\s*\(')
_URL_RE   = _re.compile(r'!\[[^\]]*\]\(\s*https?://[^\s)]+\s*\)')


def _aro_check_snippet(code):
    try:
        with _tmp.TemporaryDirectory() as t:
            (_Path(t) / 'main.aro').write_text(code)
            r = _sp.run(['aro', 'check', t], capture_output=True, text=True, timeout=10)
            return r.returncode == 0
    except Exception:
        return False


def run_eval_sweep(model_path, label):
    """Run the full eval-prompt sweep against one model. Returns a metrics
    dict (rates + per-prompt results). The model is loaded and freed inside
    so fused and quantized never occupy memory simultaneously."""
    print(f'\n[{label}] loading model from {model_path}...')
    _m, _t = load_fn(str(model_path))

    _results = []
    _replies = 0
    _empty_think = 0
    _with_code = 0
    _syntax_pass = 0
    _tool_leak = 0
    _url_contam = 0

    for _idx, _p in enumerate(_eval_prompts, 1):
        _msgs = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': _p['prompt']},
        ]
        _prompt_text = _t.apply_chat_template(_msgs, tokenize=False, add_generation_prompt=True)
        _raw = generate_fn(_m, _t, prompt=_prompt_text, max_tokens=600, verbose=False)
        _stripped = _THINK_RE.sub('', _raw).strip()
        _is_empty_think = ('<think>' in _raw and not _stripped)
        if _is_empty_think:
            _empty_think += 1
        if _stripped:
            _replies += 1
        # Content quality checks
        _blocks = _FENCE_RE.findall(_stripped)
        _has_code = bool(_blocks)
        if _has_code:
            _with_code += 1
            if all(_aro_check_snippet(b) for b in _blocks):
                _syntax_pass += 1
        if _TOOL_RE.search(_stripped):
            _tool_leak += 1
        if _URL_RE.search(_stripped):
            _url_contam += 1
        _results.append({
            'idx': _idx, 'cat': _p['cat'], 'prompt': _p['prompt'],
            'reply': _stripped[:500],
            'empty_think': _is_empty_think,
            'has_code': _has_code,
            'tool_leak': bool(_TOOL_RE.search(_stripped)),
            'url_contam': bool(_URL_RE.search(_stripped)),
        })
        if _idx % 10 == 0:
            print(f'  [{label}] [{_idx:3d}/{len(_eval_prompts)}] reply={_replies}/{_idx} '
                  f'code={_with_code} syntax-pass={_syntax_pass} '
                  f'tool-leak={_tool_leak} url-spam={_url_contam}')

    del _m
    import gc as _gc; _gc.collect()

    _N = len(_eval_prompts)
    metrics = {
        'label': label,
        'model_path': str(model_path),
        'reply_rate': _replies / _N,
        'empty_think_rate': _empty_think / _N,
        'syntax_pass_rate': _syntax_pass / _with_code if _with_code else 0.0,
        'tool_leak_rate': _tool_leak / _N,
        'url_contam_rate': _url_contam / _N,
        'total': _N,
        'with_code': _with_code,
        'results': _results,
    }
    print(f'\n[{label}] sweep results:')
    print(f'  reply rate:        {metrics["reply_rate"]:.1%}')
    print(f'  empty-think rate:  {metrics["empty_think_rate"]:.1%}')
    print(f'  code blocks:       {_with_code}/{_N}')
    print(f'  syntax pass rate:  {metrics["syntax_pass_rate"]:.1%}')
    print(f'  tool-name leak:    {metrics["tool_leak_rate"]:.1%}')
    print(f'  URL contamination: {metrics["url_contam_rate"]:.1%}')
    return metrics


# ── Save point: skip if this gate already passed for these artifacts ───────
_gate_fp = _quant_fingerprint(QUANT_DIR)
if stage_completed('promotion_gate', _gate_fp):
    _prev = stage_result('promotion_gate')
    print(f'Promotion gate already PASSED for these artifacts '
          f'(at {_prev.get("at", "?")}) — skipping the sweep.')
    for _k, _v in (_prev.get('quantized_metrics') or {}).items():
        if isinstance(_v, float):
            print(f'  {_k:<18} {_v:.1%}')
    print(f'  (set ARO_FORCE_GATES=1 to force a re-run; state: {RELEASE_STATE_PATH})')
else:
    # ── Fused (pre-quantization) sweep ──────────────────────────────────────────
    _fused_metrics = None
    if str(SOURCE_MODEL) == str(QUANT_DIR) or _is_already_quantized(SOURCE_MODEL):
        print('Source model is already quantized — fused-vs-quantized comparison '
              'is a no-op; running the gate on the quantized model only.')
    else:
        _fused_metrics = run_eval_sweep(SOURCE_MODEL, 'fused')

    # ── Quantized sweep ─────────────────────────────────────────────────────────
    _quant_metrics = run_eval_sweep(QUANT_DIR, 'quantized')

    # ── Side-by-side comparison ─────────────────────────────────────────────────
    _comparison = None
    if _fused_metrics:
        _comparison = {}
        print('\nFused vs quantized:')
        for _k, _higher_is_better in [
            ('reply_rate', True), ('empty_think_rate', False),
            ('syntax_pass_rate', True), ('tool_leak_rate', False),
            ('url_contam_rate', False),
        ]:
            _f, _q = _fused_metrics[_k], _quant_metrics[_k]
            _delta = _q - _f
            _degraded = (_delta < 0) if _higher_is_better else (_delta > 0)
            _comparison[_k] = {'fused': _f, 'quantized': _q, 'delta': _delta,
                               'degraded': bool(_degraded and abs(_delta) > 0.02)}
            _flag = '  ← quantization degradation' if _comparison[_k]['degraded'] else ''
            print(f'  {_k:<18} fused={_f:6.1%}  quant={_q:6.1%}  Δ={_delta:+.1%}{_flag}')

    # ── Persist both metric sets ────────────────────────────────────────────────
    _gate_report = QUANT_DIR / 'promotion_gate.json'
    _gate_report.write_text(_json.dumps({
        # Top-level keys mirror the quantized metrics (backward compatible).
        **{k: v for k, v in _quant_metrics.items() if k not in ('results', 'label', 'model_path')},
        'results': _quant_metrics['results'],
        'quantized': {k: v for k, v in _quant_metrics.items() if k != 'results'},
        'fused': ({k: v for k, v in _fused_metrics.items() if k != 'results'}
                  if _fused_metrics else None),
        'comparison': _comparison,
    }, indent=2))
    print(f'\n  report saved: {_gate_report}')

    # ── Gates ───────────────────────────────────────────────────────────────────
    _breaches = []
    if _quant_metrics['reply_rate'] < GATE_REPLY_RATE_MIN:
        _breaches.append(f"reply rate {_quant_metrics['reply_rate']:.1%} < {GATE_REPLY_RATE_MIN:.0%}")
    if _quant_metrics['empty_think_rate'] > GATE_EMPTY_THINK_MAX:
        _breaches.append(f"empty-think collapse {_quant_metrics['empty_think_rate']:.1%} > {GATE_EMPTY_THINK_MAX:.0%}")
    if _quant_metrics['with_code'] > 0 and _quant_metrics['syntax_pass_rate'] < GATE_SYNTAX_PASS_MIN:
        _breaches.append(f"syntax pass {_quant_metrics['syntax_pass_rate']:.1%} < {GATE_SYNTAX_PASS_MIN:.0%}")
    if _quant_metrics['tool_leak_rate'] > GATE_TOOL_LEAK_MAX:
        _breaches.append(f"tool-name leak {_quant_metrics['tool_leak_rate']:.1%} > {GATE_TOOL_LEAK_MAX:.0%} "
                         '(model emitting read_file/edit_file/aro_check as ARO)')
    if _quant_metrics['url_contam_rate'] > GATE_URL_CONTAM_MAX:
        _breaches.append(f"URL contamination {_quant_metrics['url_contam_rate']:.1%} > {GATE_URL_CONTAM_MAX:.0%} "
                         '(model emitting ![](https://...) from corpus pollution)')

    # Quantization-degradation gates — quantized must stay close to fused.
    if _fused_metrics:
        _syntax_drop = _fused_metrics['syntax_pass_rate'] - _quant_metrics['syntax_pass_rate']
        if _syntax_drop > GATE_QUANT_SYNTAX_DROP_MAX:
            _breaches.append(
                f'quantization degraded syntax pass by {_syntax_drop:.1%} '
                f'(fused {_fused_metrics["syntax_pass_rate"]:.1%} → quant '
                f'{_quant_metrics["syntax_pass_rate"]:.1%}, max drop {GATE_QUANT_SYNTAX_DROP_MAX:.0%})')
        _reply_drop = _fused_metrics['reply_rate'] - _quant_metrics['reply_rate']
        if _reply_drop > GATE_QUANT_REPLY_DROP_MAX:
            _breaches.append(
                f'quantization degraded reply rate by {_reply_drop:.1%} '
                f'(fused {_fused_metrics["reply_rate"]:.1%} → quant '
                f'{_quant_metrics["reply_rate"]:.1%}, max drop {GATE_QUANT_REPLY_DROP_MAX:.0%})')

    if _breaches:
        record_stage('promotion_gate', 'failed', fingerprint=_gate_fp,
                     breaches=_breaches,
                     quantized_metrics={k: v for k, v in _quant_metrics.items() if k != 'results'},
                     fused_metrics=({k: v for k, v in _fused_metrics.items() if k != 'results'}
                                    if _fused_metrics else None))
        raise RuntimeError(
            'PROMOTION GATE FAILED — model would ship a regression:\n  - ' +
            '\n  - '.join(_breaches) +
            f'\n\nSee {_gate_report} for per-prompt details and '
            f'{RELEASE_STATE_PATH} for the recorded gate state.'
        )

    record_stage('promotion_gate', 'passed', fingerprint=_gate_fp,
                 quantized_metrics={k: v for k, v in _quant_metrics.items() if k != 'results'},
                 fused_metrics=({k: v for k, v in _fused_metrics.items() if k != 'results'}
                                if _fused_metrics else None))
    print('\nPROMOTION GATE PASSED — model is allowed to distribute.\n')


## Step 4a — MLX server (local OpenAI-compatible API)

Writes a `serve.sh` startup script. Run it to expose the model on `http://localhost:8080/v1`
with the ARO system prompt pre-loaded.

Compatible with any OpenAI client: `curl`, Python `openai` SDK, VS Code Copilot Chat, etc.

In [ ]:
SERVE_SCRIPT = RELEASE_DIR / 'serve.sh'
CHAT_TEMPLATE_FILE = RELEASE_DIR / 'aro_system_prompt.txt'

# Write the system prompt to a standalone file beside the release scripts.
CHAT_TEMPLATE_FILE.write_text(SYSTEM_PROMPT)

# ALSO bundle the prompt into the model directory itself. The HF upload
# below uploads QUANT_DIR, so this is the only way the file ships with
# the model and lands in the user's HuggingFace cache after `aro ask`
# downloads it. ContextStore.swift `resolvedSystemPrompt(model:)` looks
# for `aro_system_prompt.txt` inside the HF cache for the configured
# model, so the runtime CLI can match the training-time prompt exactly.
_BUNDLED_PROMPT = QUANT_DIR / 'aro_system_prompt.txt'
_BUNDLED_PROMPT.write_text(SYSTEM_PROMPT)
print(f'Bundled prompt:    {_BUNDLED_PROMPT}')

# Write the startup script
serve_content = f"""#!/usr/bin/env bash
# ARO Coder — local MLX server
# Exposes an OpenAI-compatible API at http://localhost:8080/v1
#
# Usage:
#   ./serve.sh
#   curl http://localhost:8080/v1/chat/completions \\
#        -H 'Content-Type: application/json' \\
#        -d '{{"model":"aro-coder","messages":[{{"role":"user","content":"Write hello world in ARO"}}]}}'

MODEL="{QUANT_DIR}"
PORT=8080

python -m mlx_lm.server \\
    --model "$MODEL" \\
    --port  "$PORT" \\
    --system-prompt "$(cat {CHAT_TEMPLATE_FILE})"
"""
SERVE_SCRIPT.write_text(serve_content)
SERVE_SCRIPT.chmod(0o755)

print(f'MLX server script: {SERVE_SCRIPT}')
print(f'System prompt:     {CHAT_TEMPLATE_FILE}')
print()
print('To start the server:')
print(f'  {SERVE_SCRIPT}')
print()
print('To test (in another terminal):')
print("  curl http://localhost:8080/v1/chat/completions \\")
print("       -H 'Content-Type: application/json' \\")
print('       -d \'{"model":"aro-coder","messages":[{"role":"user","content":"Write hello world in ARO"}]}\'')

## Step 4b — Ollama

Creates an Ollama `Modelfile` and registers `aro-coder:latest` locally.
Requires [Ollama](https://ollama.com) to be installed and running.

Ollama uses GGUF format internally. The Modelfile points to the quantized MLX weights
via the `mlx` provider (Ollama ≥ 0.4 supports MLX models on Apple Silicon directly).

In [ ]:
MODELFILE_PATH = RELEASE_DIR / 'Modelfile'

# Build Modelfile content without nesting triple-quotes inside an f-string.
_mf_lines = [
    f'# ARO Coder — Ollama Modelfile',
    f'# Generated by notebook 24',
    f'#',
    f'# Build:  ollama create {OLLAMA_NAME} -f {MODELFILE_PATH}',
    f'# Run:    ollama run {OLLAMA_NAME}',
    f'# API:    curl http://localhost:11434/api/generate -d \'{{"model":"{OLLAMA_NAME}","prompt":"Write hello world in ARO"}}\'',
    f'',
    f'FROM {QUANT_DIR}',
    f'',
    f'SYSTEM """{SYSTEM_PROMPT}"""',
    f'',
    f'PARAMETER temperature    0.3',
    f'PARAMETER top_p          0.9',
    f'PARAMETER repeat_penalty 1.1',
    f'PARAMETER num_ctx        4096',
    f'',
    f'MESSAGE user      "Write a minimal ARO hello world application."',
    'MESSAGE assistant "(Application-Start: Hello World) {\\n    Log \\"Hello, World!\\" to the <console>.\\n    Return an <OK: status> for the <startup>.\\n}"',
]
modelfile_content = '\n'.join(_mf_lines) + '\n'

MODELFILE_PATH.write_text(modelfile_content)
print(f'Modelfile written: {MODELFILE_PATH}')

# Check model architecture — Ollama doesn't support all architectures yet
_model_arch = None
_quant_cfg = Path(QUANT_DIR) / 'config.json'
if _quant_cfg.exists():
    with open(_quant_cfg) as f:
        _cfg = json.load(f)
    _model_arch = _cfg.get('model_type', '')
    _arch_classes = _cfg.get('architectures', [])
    print(f'Model architecture: {_model_arch} ({", ".join(_arch_classes)})')

# Known unsupported architectures in Ollama (as of March 2026)
_OLLAMA_UNSUPPORTED = {'qwen3_moe', 'qwen2_moe'}

ollama_bin = shutil.which('ollama')
if not ollama_bin:
    print('\nOllama not found in PATH.')
    print('Install from https://ollama.com, then run:')
    print(f'  ollama create {OLLAMA_NAME} -f {MODELFILE_PATH}')
elif _model_arch in _OLLAMA_UNSUPPORTED:
    print(f'\n[SKIP] Ollama does not support "{_model_arch}" architecture yet.')
    print(f'Modelfile saved for future use. When Ollama adds support, run:')
    print(f'  ollama create {OLLAMA_NAME} -f {MODELFILE_PATH}')
else:
    print(f'\nOllama found: {ollama_bin}')
    print(f'Creating {OLLAMA_NAME}:latest ...')
    result = subprocess.run(
        ['ollama', 'create', OLLAMA_NAME, '-f', str(MODELFILE_PATH)],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f'\nModel registered. Run it with:')
        print(f'  ollama run {OLLAMA_NAME}')
    else:
        err = (result.stderr or result.stdout).strip()
        if 'unsupported architecture' in err.lower():
            print(f'\n[SKIP] Ollama error: {err}')
            print(f'Modelfile saved for future use. When Ollama adds support, run:')
            print(f'  ollama create {OLLAMA_NAME} -f {MODELFILE_PATH}')
        else:
            print(f'\nOllama create failed:\n  {err}')
            print('Check Ollama is running (ollama serve)')

## Step 4c — Push to HuggingFace (versioned)

Uploads the quantized model to `HF_REPO_ID` on HuggingFace.
Requires `huggingface-cli login` (run once in terminal before this cell).

Each push is a **versioned release** instead of a silent overwrite:

1. A semantic version is derived from `release/model_manifest.json`
   (same checksum → same version; new weights → minor bump; edit the
   manifest for a manual major bump).
2. The model card (README.md) is auto-generated from the manifest data,
   promotion-gate eval metrics, teacher source, quantization settings and
   known limitations.
3. The Hub commit is tagged `v<version>` so any release can be pulled back
   with `revision='v<version>'` — rollback is one argument.
4. Post-push verification re-downloads `config.json` and checks weight
   shards are present on the Hub.
5. `release/version_history.json` records every release with its gate
   metrics for attribution.

Set `HF_REPO_ID` at the top of this notebook to your org/username.


In [ ]:
try:
    from huggingface_hub import HfApi, login
except ModuleNotFoundError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
    from huggingface_hub import HfApi, login

import hashlib
from datetime import datetime, timezone

# ── Release versioning ──────────────────────────────────────────────────────
# Each push used to overwrite the previous HF revision — no tags, no
# changelog, no rollback. Releases are now semantically versioned from the
# previous release manifest, tagged on the Hub, documented by an
# auto-generated model card, and verified post-push.

def _compute_model_checksum(model_dir):
    """SHA-256 hash of all safetensors files for version fingerprinting."""
    h = hashlib.sha256()
    for f in sorted(Path(model_dir).glob('*.safetensors')):
        h.update(f.read_bytes())
    return h.hexdigest()[:16]


def _derive_release_version(manifest_path, checksum):
    """Semantic version derived from release/model_manifest.json:
      - no previous manifest              → 1.0.0
      - same checksum as previous release → previous version (idempotent re-run)
      - different checksum                → minor bump (new weights)
    For a breaking change (new base model / prompt format), bump the major
    version by editing `version` in model_manifest.json before running.
    Returns (version, previous_version_or_None)."""
    if manifest_path.exists():
        try:
            prev = json.loads(manifest_path.read_text())
            prev_version  = prev.get('version', '1.0.0')
            prev_checksum = prev.get('checksum')
            if prev_checksum == checksum:
                return prev_version, prev_version
            parts = (prev_version.split('.') + ['0', '0'])[:3]
            major, minor = int(parts[0]), int(parts[1])
            return f'{major}.{minor + 1}.0', prev_version
        except Exception as e:
            print(f'[version] Warning: could not parse previous manifest ({e}) — starting at 1.0.0')
    return '1.0.0', None


RELEASE_CHECKSUM = _compute_model_checksum(QUANT_DIR)
RELEASE_VERSION, PREV_VERSION = _derive_release_version(
    RELEASE_DIR / 'model_manifest.json', RELEASE_CHECKSUM)
print(f'Release version: v{RELEASE_VERSION}'
      + (f' (previous: v{PREV_VERSION})' if PREV_VERSION else ' (first release)'))


def _load_gate_metrics():
    """Promotion-gate metrics (quantized + fused) for the model card."""
    _p = QUANT_DIR / 'promotion_gate.json'
    if _p.exists():
        try:
            return json.loads(_p.read_text())
        except Exception:
            pass
    return {}


def build_model_card(repo_id):
    """Auto-generate the model card from the manifest data: version, teacher
    source, quantization settings, promotion-gate eval metrics, known
    limitations and version history."""
    gate  = _load_gate_metrics()
    quant = gate.get('quantized') or {}
    fused = gate.get('fused') or {}

    _stats_path = DATA_ROOT / '05_dataset' / 'stats.json'
    if _stats_path.exists():
        with open(_stats_path) as _f:
            _ds_stats = json.load(_f)
    else:
        _ds_stats = {'total': '?', 'train': '?', 'valid': '?', 'test': '?'}

    def _pct(d, key):
        v = d.get(key)
        return f'{v:.1%}' if isinstance(v, (int, float)) else '?'

    # Evaluation table — quantized, with the fused column when the
    # fused-vs-quantized comparison ran.
    if quant:
        if fused:
            eval_md = f"""## Evaluation (promotion gate, {quant.get('total', '?')} prompts)

| Metric | Quantized (shipped) | Fused (pre-quantization) |
|---|---|---|
| Reply rate | {_pct(quant, 'reply_rate')} | {_pct(fused, 'reply_rate')} |
| Empty-think collapse | {_pct(quant, 'empty_think_rate')} | {_pct(fused, 'empty_think_rate')} |
| Syntax pass rate (`aro check`) | {_pct(quant, 'syntax_pass_rate')} | {_pct(fused, 'syntax_pass_rate')} |
| Tool-name leakage | {_pct(quant, 'tool_leak_rate')} | {_pct(fused, 'tool_leak_rate')} |
| URL contamination | {_pct(quant, 'url_contam_rate')} | {_pct(fused, 'url_contam_rate')} |
"""
        else:
            eval_md = f"""## Evaluation (promotion gate, {quant.get('total', '?')} prompts)

| Metric | Quantized (shipped) |
|---|---|
| Reply rate | {_pct(quant, 'reply_rate')} |
| Empty-think collapse | {_pct(quant, 'empty_think_rate')} |
| Syntax pass rate (`aro check`) | {_pct(quant, 'syntax_pass_rate')} |
| Tool-name leakage | {_pct(quant, 'tool_leak_rate')} |
| URL contamination | {_pct(quant, 'url_contam_rate')} |
"""
    else:
        eval_md = '## Evaluation\n\nPromotion-gate metrics were not found for this build.\n'

    # Version history — current release first, then prior entries on file.
    _rows = [f"| v{RELEASE_VERSION} | {datetime.now(timezone.utc).date().isoformat()} "
             f"| {SOURCE_LABEL} | `{RELEASE_CHECKSUM}` |"]
    _hist_path = RELEASE_DIR / 'version_history.json'
    if _hist_path.exists():
        try:
            _hist = json.loads(_hist_path.read_text())
            for h in reversed(_hist[-9:]):
                if h.get('version') == RELEASE_VERSION:
                    continue
                _rows.append(f"| v{h.get('version')} | {str(h.get('pushed_at', ''))[:10]} "
                             f"| {h.get('source_label', '?')} | `{h.get('checksum', '?')}` |")
        except Exception:
            pass
    history_md = ('## Version History\n\n'
                  '| Version | Date | Source | Checksum |\n|---|---|---|---|\n'
                  + '\n'.join(_rows) + '\n\n'
                  'Every release is tagged on the Hub — load an older version with\n'
                  f'`load("{repo_id}", revision="v<version>")` or report issues against\n'
                  'the version shown by `aro ask --version`.\n')

    return f"""---
language: en
license: mit
tags:
  - aro
  - code-generation
  - dsl
  - mlx
  - 4-bit
  - lora
  - fine-tuned
base_model: {MODEL_ID}
pipeline_tag: text-generation
library_name: mlx
---

# ARO Coder — v{RELEASE_VERSION}

A fine-tuned code generation model specialised in the **ARO** (Action Result Object) programming language.

ARO is a domain-specific language where every statement follows the pattern:
`Verb the <Result> preposition [the] <Object>`.

| | |
|---|---|
| **Version** | v{RELEASE_VERSION} (tag `v{RELEASE_VERSION}`) |
| **Checksum** | `{RELEASE_CHECKSUM}` |
| **Base model** | [{MODEL_ID}](https://huggingface.co/{MODEL_ID}) |
| **Teacher source** | {SOURCE_LABEL} (30B MoE teacher distilled to 8B student) |
| **Quantization** | 4-bit MLX, group size 64 |
| **Language** | ARO |
| **Training samples** | {_ds_stats.get('total', '?')} |

## Links

- **Website**: [arolang.github.io/aro](https://arolang.github.io/aro/)
- **GitHub**: [github.com/arolang/aro](https://github.com/arolang/aro)
- **Documentation**: [Wiki](https://github.com/arolang/aro/wiki)
- **Language Guide (PDF)**: [Download](https://github.com/arolang/aro/releases/latest/download/ARO-Language-Guide.pdf)
- **Discussions**: [GitHub Discussions](https://github.com/arolang/aro/discussions)

{eval_md}
## Known Limitations

- **Happy-path DSL only** — ARO code deliberately contains no error handling;
  do not expect defensive code from this model.
- **4-bit quantization** — small quality loss vs the fused model is expected;
  the promotion gate bounds the degradation (see the table above when both
  columns are present).
- **Verb hallucination at high temperatures** — keep temperature ≤ 0.3 for
  code generation; the model may invent non-existent action verbs above that.
- **English-only** instructions and answers.
- Knowledge is frozen at training time; language features newer than this
  release's corpus are unknown to the model.

## Quick Start

### MLX (Apple Silicon)

```python
from mlx_lm import load, generate

model, tokenizer = load("{repo_id}")            # latest release
# model, tokenizer = load("{repo_id}", revision="v{RELEASE_VERSION}")  # pinned

messages = [
    {{"role": "system", "content": "You are an expert ARO programmer."}},
    {{"role": "user", "content": "Write an ARO feature set that retrieves a user by ID and returns an OK response."}},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
response = generate(model, tokenizer, prompt=prompt, max_tokens=500)
print(response)
```

### MLX Server (OpenAI-compatible API)

```bash
python -m mlx_lm.server --model {repo_id} --port 8080

curl http://localhost:8080/v1/chat/completions \\
  -H 'Content-Type: application/json' \\
  -d '{{"model": "aro-coder", "messages": [{{"role": "user", "content": "Write hello world in ARO"}}]}}'
```

### Ollama

```bash
ollama run aro-coder
```

## Example Output

**Prompt:** *Write an ARO Application-Start that starts an HTTP server.*

```aro
(Application-Start: My API) {{
    Log "Starting server..." to the <console>.
    Start the <http-server> with <contract>.
    Keepalive the <application> for the <events>.
    Return an <OK: status> for the <startup>.
}}
```

## What is ARO?

ARO is a DSL for expressing business features as Action-Result-Object statements.
Every program is a directory of `.aro` files with event-driven feature sets:

```aro
(getUser: User API) {{
    Extract the <id> from the <pathParameters: id>.
    Retrieve the <user> from the <user-repository> where id = <id>.
    Return an <OK: status> with <user>.
}}
```

Key features:
- **Contract-first HTTP** — routes defined in `openapi.yaml`, feature sets match `operationId`
- **Event-driven** — feature sets triggered by events, not direct calls
- **Immutable bindings** — every transformation produces a new name
- **Happy-path only** — no error handling code; the runtime manages errors

## Training

This model was trained with the ARO training pipeline:

1. **Corpus collection** — {_ds_stats.get('total', '?')} samples from Examples, Book, Wiki, Proposals, and real-world ARO applications
2. **Supervised fine-tuning** — LoRA on all code generation, debugging, Q&A, and explanation tasks
3. **DPO preference training** — using `aro check` validation to build chosen/rejected pairs
4. **Iterative self-improvement** — multiple rounds of generate-validate-retrain
5. **Distillation** — the 30B MoE teacher's outputs (syntax- and semantically-gated) train the 8B student
6. **Promotion gate** — 100-prompt sweep on both fused and quantized weights before any distribution

{history_md}
## License

This model and the ARO language are open source under the [MIT License](https://github.com/arolang/aro/blob/main/LICENSE).
"""


def _verify_push(repo_id):
    """Post-push verification: the model must be downloadable and
    structurally loadable — config.json re-downloads and parses, weight
    shards and tokenizer files are present on the Hub. (A full mlx load
    would re-download the whole model; NB26 post-release validation does
    that as its own pipeline stage.)"""
    from huggingface_hub import hf_hub_download
    api = HfApi()
    info = api.model_info(repo_id)
    names = {s.rfilename for s in (info.siblings or [])}
    missing = [n for n in ('config.json',) if n not in names]
    if not any(n.endswith('.safetensors') for n in names):
        missing.append('*.safetensors')
    if missing:
        raise RuntimeError(f'Post-push verification FAILED: {repo_id} is missing {missing}')
    cfg_path = hf_hub_download(repo_id, 'config.json', force_download=True)
    with open(cfg_path) as f:
        json.load(f)
    _shards = sum(1 for n in names if n.endswith('.safetensors'))
    print(f'✓ Post-push verification: config.json downloads + parses, '
          f'{_shards} weight shard(s) present')


def _append_version_history(repo_id, commit):
    """Record this release in release/version_history.json for rollback and
    attribution."""
    gate  = _load_gate_metrics()
    quant = gate.get('quantized') or {}
    _hist_path = RELEASE_DIR / 'version_history.json'
    hist = []
    if _hist_path.exists():
        try:
            hist = json.loads(_hist_path.read_text())
        except Exception:
            hist = []
    hist = [h for h in hist if h.get('version') != RELEASE_VERSION]  # idempotent re-push
    hist.append({
        'version': RELEASE_VERSION,
        'previous_version': PREV_VERSION,
        'checksum': RELEASE_CHECKSUM,
        'repo_id': repo_id,
        'hub_commit': getattr(commit, 'oid', None),
        'hub_tag': f'v{RELEASE_VERSION}',
        'source_label': SOURCE_LABEL,
        'base_model': str(MODEL_ID),
        'pushed_at': datetime.now(timezone.utc).isoformat(),
        'gate_metrics': {k: quant.get(k) for k in
                         ('reply_rate', 'empty_think_rate', 'syntax_pass_rate',
                          'tool_leak_rate', 'url_contam_rate')},
    })
    _hist_path.write_text(json.dumps(hist, indent=2))
    print(f'Version history updated: {_hist_path} ({len(hist)} release(s))')


def push_to_hf(local_dir, repo_id, private=False):
    api = HfApi()

    # Create repo if it doesn't exist
    try:
        api.create_repo(repo_id=repo_id, repo_type='model', private=private, exist_ok=True)
        print(f'Repo ready: https://huggingface.co/{repo_id}')
    except Exception as e:
        print(f'Could not create repo: {e}')
        print('Make sure you are logged in: huggingface-cli login')
        return

    # ── Auto-generated model card ─────────────────────────────────────────
    readme = build_model_card(repo_id)
    (Path(local_dir) / 'README.md').write_text(readme)
    print(f'Model card written ({len(readme):,} chars)')

    # Clean old files from remote before uploading to prevent stale weight shards
    print(f'Uploading {local_dir} → {repo_id} (release v{RELEASE_VERSION})...')
    commit = api.upload_folder(
        folder_path=str(local_dir),
        repo_id=repo_id,
        repo_type='model',
        commit_message=f'Release v{RELEASE_VERSION} — ARO Coder 4-bit ({SOURCE_LABEL}, {RELEASE_CHECKSUM})',
        delete_patterns=["*.safetensors", "*.json", "*.py", "*.txt", "*.model",
                         "*.tiktoken", ".source_model"],
    )
    print(f'\n✓ Uploaded to https://huggingface.co/{repo_id}')

    # ── Tag the release on the Hub (rollback point) ───────────────────────
    try:
        api.create_tag(
            repo_id=repo_id, repo_type='model',
            tag=f'v{RELEASE_VERSION}',
            revision=getattr(commit, 'oid', None),
            tag_message=f'ARO Coder v{RELEASE_VERSION} ({SOURCE_LABEL}, checksum {RELEASE_CHECKSUM})',
        )
        print(f'✓ Tagged release: v{RELEASE_VERSION}')
    except Exception as e:
        # An identical re-push of the same weights keeps the same version —
        # the tag then already exists, which is fine.
        print(f'Tag v{RELEASE_VERSION} not created ({e!s:.120}) — '
              'expected on an idempotent re-push of the same weights.')

    # ── Post-push verification + version history ─────────────────────────
    _verify_push(repo_id)
    _append_version_history(repo_id, commit)
    return commit


# Auto-skip when HF_TOKEN is missing — the original always-on
# call hung NB24 for 2 hours waiting on credentials it never got.
# Resolve HF auth from either HF_TOKEN env or the cached token file
# (huggingface-cli login). Pushes only fire when one of the two
# returns a usable identity, so this won't hang waiting on creds.
import os
# Use the modern token resolver where available, fall back to HfFolder.
from huggingface_hub import whoami
try:
    from huggingface_hub import get_token as _hf_get_token
except ImportError:
    from huggingface_hub import HfFolder
    _hf_get_token = HfFolder.get_token
_hf_token = os.environ.get('HF_TOKEN') or _hf_get_token()
_hf_ok = False
if _hf_token:
    try:
        _info = whoami(token=_hf_token)
        print(f'HF auth: {_info.get("name")} (orgs: {[o.get("name") for o in _info.get("orgs", [])]})')
        _hf_ok = True
    except Exception as _e:
        print(f'HF auth failed ({_e!s:.120}). Skipping upload.')
if _hf_ok:
    os.environ['HF_TOKEN'] = _hf_token  # huggingface_hub picks this up implicitly
    push_to_hf(QUANT_DIR, HF_REPO_ID)
else:
    print('No HF token (env or ~/.cache/huggingface/token) — skipping HuggingFace upload.')
    print(f'To publish later: push_to_hf(QUANT_DIR, "{HF_REPO_ID}")')

print('HuggingFace push ready.')
print(f'To publish, call: push_to_hf(QUANT_DIR, "{HF_REPO_ID}")')
print('(Make sure you ran: huggingface-cli login)')


## Step 4d — Push Teacher Model to HuggingFace

Uploads the full fine-tuned 30B teacher model to `ARO-Lang/aro-teacher-30b-4bit`.
This preserves the teacher for future training cycles — the next pipeline run
can start from this model instead of vanilla Qwen.

The teacher model is ~16 GB. Upload may take a while.

In [ ]:
HF_TEACHER_REPO_ID = 'ARO-Lang/aro-teacher-30b-4bit'

def find_best_teacher_model():
    """Find the best locally-fused 30B teacher model.
    Priority: DPO > iterative (highest round) > finetune (highest round)."""
    # DPO
    dpo = MODELS_DIR / 'dpo' / 'fused'
    if dpo.exists() and list(dpo.glob('*.safetensors')):
        return dpo, 'dpo'
    # Iterative (highest round)
    it_dir = MODELS_DIR / 'iterative'
    if it_dir.exists():
        rounds = sorted(it_dir.glob('round_*/fused'), key=lambda p: p.parent.name)
        for r in reversed(rounds):
            if list(r.glob('*.safetensors')):
                return r, f'iterative_{r.parent.name}'
    # Fine-tune (highest round)
    ft_dir = MODELS_DIR / 'finetune'
    if ft_dir.exists():
        rounds = sorted(ft_dir.glob('round_*/fused'), key=lambda p: p.parent.name)
        for r in reversed(rounds):
            if list(r.glob('*.safetensors')):
                return r, f'finetune_{r.parent.name}'
    return None, None


def push_teacher_to_hf(teacher_dir, repo_id):
    """Upload the 30B teacher model with a dedicated README."""
    api = HfApi()
    try:
        api.create_repo(repo_id=repo_id, repo_type='model', private=False, exist_ok=True)
        print(f'Repo ready: https://huggingface.co/{repo_id}')
    except Exception as e:
        print(f'Could not create repo: {e}')
        return

    readme = f"""---
language: en
license: mit
tags:
  - aro
  - code-generation
  - dsl
  - mlx
  - 4-bit
  - teacher-model
  - fine-tuned
base_model: mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit
pipeline_tag: text-generation
library_name: mlx
---

# ARO Teacher (30B)

The full **30B MoE teacher model**, fine-tuned on ARO training data.
This model is used for:

- **Distillation** \u2014 generating high-quality training data for the smaller student model
- **Iterative retraining** \u2014 serving as the starting point for the next training cycle
- **High-quality inference** \u2014 when maximum accuracy is needed (at the cost of speed/memory)

> **For deployment and everyday use, prefer the distilled 8B student model:**
> [ARO-Lang/aro-coder-4bit](https://huggingface.co/ARO-Lang/aro-coder-4bit)

| | |
|---|---|
| **Architecture** | Qwen3 30B MoE (3.3B active parameters) |
| **Base model** | [mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit](https://huggingface.co/mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit) |
| **Quantization** | 4-bit (MLX) |
| **Size** | ~16 GB |
| **Training source** | {teacher_label} |

## Usage

```python
from mlx_lm import load, generate
model, tokenizer = load(\"{repo_id}\")
```

Or as a base for continued fine-tuning:

```bash
python -m mlx_lm lora --model {repo_id} --data ./train_data --train
```

## Links

- **Distilled student**: [ARO-Lang/aro-coder-4bit](https://huggingface.co/ARO-Lang/aro-coder-4bit)
- **Website**: [arolang.github.io/aro](https://arolang.github.io/aro/)
- **GitHub**: [github.com/arolang/aro](https://github.com/arolang/aro)
- **Language Guide**: [Wiki](https://github.com/arolang/aro/wiki)

## License

MIT License
"""
    (Path(teacher_dir) / 'README.md').write_text(readme)
    print(f'Teacher README.md written')

    size_gb = sum(f.stat().st_size for f in Path(teacher_dir).rglob('*') if f.is_file()) / 1e9
    print(f'Uploading teacher model (~{size_gb:.1f} GB) to {repo_id}...')
    api.upload_folder(
        folder_path=str(teacher_dir),
        repo_id=repo_id,
        repo_type='model',
        commit_message=f'Upload ARO Teacher 30B ({teacher_label})',
        delete_patterns=['*.safetensors', '*.json', '*.py', '*.txt', '*.model',
                         '*.tiktoken', '.source_model', '*.md'],
    )
    print(f'\\n\u2713 Teacher uploaded to https://huggingface.co/{repo_id}')


teacher_path, teacher_label = find_best_teacher_model()
if teacher_path:
    print(f'Best teacher model: {teacher_path} ({teacher_label})')
    if _hf_ok:
        push_teacher_to_hf(teacher_path, HF_TEACHER_REPO_ID)
    else:
        print('No HF auth — skipping teacher upload.')
else:
    print('No fused teacher model found \u2014 skipping teacher upload.')
    print('  Run NB12 (finetune) first to produce a fused teacher model.')


## Step 5 — `aro ask` version manifest

Writes a `model_manifest.json` into the release directory. The `aro ask` CLI command
reads this manifest to check whether the local model is up-to-date and to prompt the
user before downloading a new version.

In [ ]:
from datetime import datetime, timezone
import hashlib

MANIFEST_PATH = RELEASE_DIR / 'model_manifest.json'

def _compute_model_checksum(model_dir):
    """SHA-256 hash of all safetensors files for version fingerprinting."""
    h = hashlib.sha256()
    model_dir = Path(model_dir)
    for f in sorted(model_dir.glob('*.safetensors')):
        h.update(f.read_bytes())
    return h.hexdigest()[:16]

_checksum = _compute_model_checksum(QUANT_DIR)
_size_bytes = sum(f.stat().st_size for f in Path(QUANT_DIR).rglob('*') if f.is_file())

manifest = {
    'model_id': HF_REPO_ID,
    'source_label': SOURCE_LABEL,
    'base_model': str(MODEL_ID),
    'quantization': '4-bit',
    'checksum': _checksum,
    'size_bytes': _size_bytes,
    'built_at': datetime.now(timezone.utc).isoformat(),
    'cli_command': 'aro ask',
    'min_cli_version': '1.0.0',
}

with open(MANIFEST_PATH, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'Model manifest written: {MANIFEST_PATH}')
print(f'  Checksum:  {_checksum}')
print(f'  Size:      {_size_bytes / 1e9:.1f} GB')
print(f'  Built at:  {manifest["built_at"]}')
print()
print('The `aro ask` CLI reads this manifest to:')
print('  1. Check if the model is already downloaded')
print('  2. Compare checksums to detect newer versions')
print('  3. Prompt the user before downloading/updating')

## Step 6 — Release summary

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222', 'axes.labelcolor': '#222222',
    'xtick.color': '#333333', 'ytick.color': '#333333',
    'axes.titlecolor': '#111111', 'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9', 'savefig.facecolor': '#fafafa',
    'savefig.bbox': 'tight', 'savefig.dpi': 150,
})
from pathlib import Path
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '22_package.png'

# Collect file sizes from quantized model
_file_sizes = {}
if QUANT_DIR.exists():
    for f in sorted(QUANT_DIR.rglob('*')):
        if f.is_file():
            ext = f.suffix or f.name
            _file_sizes[ext] = _file_sizes.get(ext, 0) + f.stat().st_size

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5))

# ── Left: Model file sizes ───────────────────────────────────────────
if _file_sizes:
    _ext_sorted = sorted(_file_sizes.items(), key=lambda x: -x[1])
    _ext_labels = [e for e, _ in _ext_sorted]
    _ext_gb = [s / 1e9 for _, s in _ext_sorted]
    bars = ax1.barh(_ext_labels, _ext_gb, color='#3498db', edgecolor='white', height=0.5)
    for bar, gb in zip(bars, _ext_gb):
        label = f'{gb:.2f} GB' if gb >= 0.01 else f'{gb*1000:.1f} MB'
        ax1.text(bar.get_width() + max(_ext_gb)*0.02, bar.get_y() + bar.get_height()/2,
                 label, va='center', fontsize=9)
    ax1.set_xlabel('Size (GB)')
    ax1.invert_yaxis()
else:
    ax1.text(0.5, 0.5, 'no model files', ha='center', va='center', transform=ax1.transAxes)
ax1.set_title('Model File Sizes', fontsize=12, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

# ── Middle: Smoke test results ───────────────────────────────────────
if results:
    _smoke_labels = [r['prompt'][:30] + '...' for r in results]
    _smoke_colors = ['#2ecc71' if r['aro_check_pass'] else
                     ('#e74c3c' if r['has_code_block'] else '#95a5a6')
                     for r in results]
    _smoke_vals = [1 for _ in results]
    bars_smoke = ax2.barh(_smoke_labels, _smoke_vals, color=_smoke_colors, edgecolor='white', height=0.5)
    for bar, r in zip(bars_smoke, results):
        label = 'PASS' if r['aro_check_pass'] else ('FAIL' if r['has_code_block'] else 'NO CODE')
        ax2.text(0.5, bar.get_y() + bar.get_height()/2, label,
                 va='center', ha='center', fontsize=10, fontweight='bold', color='white')
    ax2.set_xlim(0, 1.2)
    ax2.set_xticks([])
    ax2.invert_yaxis()
    ax2.set_title(f'Smoke Test ({passed}/{len(results)} pass)', fontsize=12, fontweight='bold')
else:
    ax2.text(0.5, 0.5, 'no smoke test data', ha='center', va='center', transform=ax2.transAxes)

# ── Right: Distribution targets checklist ────────────────────────────
_targets = []
if SERVE_SCRIPT.exists(): _targets.append(('MLX Server', True))
else: _targets.append(('MLX Server', False))
if MODELFILE_PATH.exists(): _targets.append(('Ollama', True))
else: _targets.append(('Ollama', False))
if MANIFEST_PATH.exists(): _targets.append(('aro ask', True))
else: _targets.append(('aro ask', False))
_targets.append(('HuggingFace', False))

_t_labels = [t[0] for t in _targets]
_t_vals   = [1 if t[1] else 0.3 for t in _targets]
_t_colors = ['#2ecc71' if t[1] else '#e0e0e0' for t in _targets]
bars2 = ax3.barh(_t_labels, _t_vals, color=_t_colors, edgecolor='white', height=0.5)
for bar, (name, ready) in zip(bars2, _targets):
    ax3.text(0.5, bar.get_y() + bar.get_height()/2,
             'Ready' if ready else 'Pending',
             va='center', ha='center', fontsize=11, fontweight='bold',
             color='white' if ready else '#999')
ax3.set_xlim(0, 1.2)
ax3.set_xticks([])
ax3.invert_yaxis()
ax3.set_title('Distribution Targets', fontsize=12, fontweight='bold')

_total_gb = sum(_file_sizes.values()) / 1e9 if _file_sizes else 0
fig.suptitle(
    f'Package \u2014 {SOURCE_LABEL}  \u00b7  {_total_gb:.1f} GB total',
    fontsize=13, fontweight='bold', y=1.02,
)
fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')

In [ ]:
print('=' * 60)
print('ARO Coder — Release Summary')
print('=' * 60)
print(f'Source model:    {SOURCE_MODEL}')
print(f'Source label:    {SOURCE_LABEL}')
print(f'Base model:      {MODEL_ID}')
print()

if QUANT_DIR.exists():
    size_gb = sum(f.stat().st_size for f in QUANT_DIR.rglob('*') if f.is_file()) / 1e9
    print(f'Quantized model: {QUANT_DIR}')
    print(f'  Size:          {size_gb:.1f} GB')
print()

print('Distribution targets:')

if SERVE_SCRIPT.exists():
    print(f'  [✓] MLX server  →  {SERVE_SCRIPT}')
    print(f'      Start:  bash {SERVE_SCRIPT}')
    print(f'      API:    http://localhost:8080/v1/chat/completions')
print()

if MODELFILE_PATH.exists():
    ollama_ok = shutil.which('ollama') is not None
    status = '✓' if ollama_ok else '○ (Ollama not installed)'
    print(f'  [{status}] Ollama      →  {MODELFILE_PATH}')
    print(f'      Build:  ollama create {OLLAMA_NAME} -f {MODELFILE_PATH}')
    print(f'      Run:    ollama run {OLLAMA_NAME}')
print()

print(f'  [○] HuggingFace →  https://huggingface.co/{HF_REPO_ID}')
print(f'      Publish: push_to_hf(QUANT_DIR, "{HF_REPO_ID}")')
print()

if MANIFEST_PATH.exists():
    print(f'  [✓] aro ask     →  {MANIFEST_PATH}')
    print(f'      Usage:  aro ask "Write an ARO hello world app"')
    print(f'      Usage:  aro ask "How does the Publish action work?"')
print()